# CVE Data Ingestion & Reset Pipeline

**Purpose**: Complete data pipeline for resetting and reloading CVE database with all enrichments

**What this notebook does**:
1. **Status Check** - View current database and cache status
2. **Reset Options** - Clear database, cache, or both
3. **Data Ingestion** - Fetch CVEs from NVD API
4. **Enrichment** - Add KEV, EPSS, Healthcare, ATT&CK, CHPL data
5. **Validation** - Verify data quality and completeness

**Prerequisites**:
- Set `NVD_API_KEY` in .env file (optional but recommended for faster fetching)
- Ensure all required cache directories exist

---

## 1. Setup & Imports

In [9]:
import sys
import os
from pathlib import Path
import pandas as pd
import sqlite3
import shutil
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Import project modules
from src.core.cve_database import CVEDatabase
from src.core import cti_recommender
from src.core.epss_fetcher import EPSSFetcher
from src.core.healthcare_curated import HealthcareCuratedDataset
from src.analysis.healthcare_mapping import HealthcareMapper
from src.analysis.attack_mapper import AttackMapper
from src.analysis.chpl_mapper import CHPLMapper
from config.settings import settings

print(f"[OK] Project root: {project_root}")
print(f"[OK] Database path: {settings.get_database_path()}")
print(f"[OK] Imports successful")

[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Database path: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
[OK] Imports successful


## 2. Current Status Check

View current state of database and cache files

In [10]:
def check_current_status():
    """Display comprehensive status of database and cache"""
    
    print("="*80)
    print("CURRENT DATA STATUS")
    print("="*80)
    
    # Database status
    db_path = project_root / "data" / "cve_database.db"
    if db_path.exists():
        db_size = db_path.stat().st_size / (1024**2)  # MB
        print(f"\n[STATS] Database: {db_path}")
        print(f"   Size: {db_size:.2f} MB")
        
        # Query stats
        db = CVEDatabase(db_path)
        cursor = db.conn.cursor()
        
        cursor.execute('SELECT COUNT(*) FROM cves')
        cve_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM enrichments')
        enrichment_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM enrichments WHERE kev_flag = 1')
        kev_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM enrichments WHERE chpl_flag = 1')
        chpl_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM enrichments WHERE is_healthcare = 1')
        healthcare_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT COUNT(*) FROM enrichments WHERE attack_flag = 1')
        attack_count = cursor.fetchone()[0]
        
        cursor.execute('SELECT MIN(published), MAX(published) FROM cves')
        date_range = cursor.fetchone()
        
        print(f"   Total CVEs: {cve_count:,}")
        print(f"   Enriched: {enrichment_count:,}")
        print(f"   Date range: {date_range[0]} to {date_range[1]}")
        print(f"\n   Enrichment Signals:")
        print(f"      KEV (exploited): {kev_count:,}")
        print(f"      CHPL certified: {chpl_count:,}")
        print(f"      Healthcare: {healthcare_count:,}")
        print(f"      ATT&CK mapped: {attack_count:,}")
        
        db.conn.close()
    else:
        print(f"\n[WARN]  Database not found: {db_path}")
    
    # Cache status
    print(f"\n Cache Directories:")
    cache_dirs = ['cache/nvd', 'cache/epss', 'cache/kev', 'cache/attack', 'cache/chpl']
    for cache_dir in cache_dirs:
        cache_path = project_root / cache_dir
        if cache_path.exists():
            files = list(cache_path.glob('*'))
            total_size = sum(f.stat().st_size for f in files if f.is_file()) / (1024**2)
            print(f"   {cache_dir}: {len(files)} files ({total_size:.2f} MB)")
        else:
            print(f"   {cache_dir}: Not found")
    
    print("\n" + "="*80)

check_current_status()

CURRENT DATA STATUS

[STATS] Database: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
   Size: 322.10 MB
2026-02-26 22:58:14 - src.core.cve_database - INFO - Connected to database
2026-02-26 22:58:14 - src.core.cve_database - INFO - Database schema created/verified
   Total CVEs: 226,320
   Enriched: 226,320
   Date range: 2018-01-01T00:29:00.213 to 2025-12-31T23:15:42.413

   Enrichment Signals:
      KEV (exploited): 1,179
      CHPL certified: 5,107
      Healthcare: 822
      ATT&CK mapped: 83,574

 Cache Directories:
   cache/nvd: 3 files (0.70 MB)
   cache/epss: 2 files (21.56 MB)
   cache/kev: 1 files (0.12 MB)
   cache/attack: 1 files (0.67 MB)
   cache/chpl: 2 files (4.13 MB)



## 3. Reset Options

[WARN] **Warning**: These operations are destructive and cannot be undone!

In [3]:
def reset_database(confirm: bool = False):
    """Delete and recreate database"""
    if not confirm:
        print("[WARN]  Set confirm=True to actually delete the database")
        return
    
    db_path = project_root / "data" / "cve_database.db"
    if db_path.exists():
        db_path.unlink()
        print(f"[OK] Deleted database: {db_path}")
    
    # Recreate empty database
    db = CVEDatabase(db_path)
    db.conn.close()
    print(f"[OK] Created fresh database")

def reset_cache(cache_type: str = 'all', confirm: bool = False):
    """Clear cache directories
    
    Args:
        cache_type: 'all', 'nvd', 'epss', 'kev', 'attack', or 'chpl'
        confirm: Must be True to actually delete
    """
    if not confirm:
        print("[WARN]  Set confirm=True to actually delete cache")
        return
    
    cache_dirs = {
        'nvd': 'cache/nvd',
        'epss': 'cache/epss',
        'kev': 'cache/kev',
        'attack': 'cache/attack',
        'chpl': 'cache/chpl'
    }
    
    if cache_type == 'all':
        dirs_to_clear = cache_dirs.values()
    elif cache_type in cache_dirs:
        dirs_to_clear = [cache_dirs[cache_type]]
    else:
        print(f"[WARN]  Unknown cache type: {cache_type}")
        return
    
    for cache_dir in dirs_to_clear:
        cache_path = project_root / cache_dir
        if cache_path.exists():
            shutil.rmtree(cache_path)
            cache_path.mkdir(parents=True, exist_ok=True)
            print(f"[OK] Cleared: {cache_dir}")

def reset_all(confirm: bool = False):
    """Reset database AND all cache"""
    if not confirm:
        print("[WARN]  Set confirm=True to reset everything")
        return
    
    reset_database(confirm=True)
    reset_cache(cache_type='all', confirm=True)
    print("\n[OK] Complete reset finished")

# Example usage (uncomment to use):
# reset_database(confirm=True)
# reset_cache(cache_type='epss', confirm=True)
# reset_all(confirm=True)

print("Reset functions loaded. Use with confirm=True to execute.")

Reset functions loaded. Use with confirm=True to execute.


## 4. Fetch CVEs from NVD

Fetch CVE data for a specific date range

In [4]:
def fetch_cves_by_date(start_date: str, end_date: str, api_key: str = None):
    """
    Fetch CVEs for date range and insert into database
    
    Args:
        start_date: Start date (YYYY-MM-DD)
        end_date: End date (YYYY-MM-DD)
        api_key: NVD API key (optional, uses env var if not provided)
    
    Returns:
        DataFrame of fetched CVEs
    """
    if api_key is None:
        api_key = os.environ.get("NVD_API_KEY")
    
    print(f"\n{'='*80}")
    print(f"FETCHING CVEs: {start_date} to {end_date}")
    print(f"{'='*80}")
    
    if api_key:
        print("[OK] Using NVD API key (faster rate limit)")
    else:
        print("[WARN]  No API key - using slower rate limit (6s per request)")
    
    # Fetch data
    print(f"\nFetching from NVD API...")
    df = cti_recommender.fetch_nvd_date_range(
        start_date=start_date,
        end_date=end_date,
        api_key=api_key
    )
    
    if df.empty:
        print("[WARN]  No CVEs found for this date range")
        return df
    
    print(f"\n[OK] Fetched {len(df):,} CVEs")
    
    # Insert into database
    db_path = project_root / "data" / "cve_database.db"
    db = CVEDatabase(db_path)
    
    print(f"\nInserting into database...")
    count = db.upsert_cves(df)
    
    # Log the fetch
    db.log_fetch(
        start_date=start_date,
        end_date=end_date,
        cve_count=count,
        fetch_type='manual',
        status='success'
    )
    
    db.conn.close()
    print(f"[OK] Inserted {count:,} CVEs into database")
    print(f"{'='*80}\n")
    
    return df

# Example: Fetch last 30 days
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=30)

print(f"Ready to fetch CVEs from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print("\nTo fetch, run:")
print(f"df = fetch_cves_by_date('{start_date.strftime('%Y-%m-%d')}', '{end_date.strftime('%Y-%m-%d')}')")

Ready to fetch CVEs from 2026-01-24 to 2026-02-23

To fetch, run:
df = fetch_cves_by_date('2026-01-24', '2026-02-23')


### Quick Fetch Options

Uncomment one of the following to fetch data:

In [5]:
# Option 1: Fetch last 7 days
# end = datetime.now(timezone.utc).strftime('%Y-%m-%d')
# start = (datetime.now(timezone.utc) - timedelta(days=7)).strftime('%Y-%m-%d')
# df = fetch_cves_by_date(start, end)

# Option 2: Fetch last 30 days
# end = datetime.now(timezone.utc).strftime('%Y-%m-%d')
# start = (datetime.now(timezone.utc) - timedelta(days=30)).strftime('%Y-%m-%d')
# df = fetch_cves_by_date(start, end)

# Option 3: Fetch specific date range
# df = fetch_cves_by_date('2024-01-01', '2024-01-31')

# Option 4: Fetch 2025 data (current year)
# df = fetch_cves_by_date('2025-01-01', '2025-12-31')

print("Uncomment one of the options above to fetch CVEs")

Uncomment one of the options above to fetch CVEs


## 5. Enrich CVEs with Multi-Source Data

Add KEV, EPSS, Healthcare, ATT&CK, and CHPL enrichments

In [6]:
import requests

def enrich_all_cves():
    """
    Comprehensive enrichment pipeline:
    1. KEV flags
    2. EPSS scores
    3. Healthcare mapping
    4. ATT&CK mapping
    5. CHPL mapping
    """
    
    db_path = project_root / "data" / "cve_database.db"
    db = CVEDatabase(db_path)
    
    print("="*80)
    print("ENRICHMENT PIPELINE")
    print("="*80)
    
    # Get all CVEs
    df = db.get_all_cves()
    print(f"\n[STATS] Total CVEs to enrich: {len(df):,}")
    
    # ========================================================================
    # 1. KEV FLAGS
    # ========================================================================
    print("\n[1/5] Fetching CISA KEV catalog...")
    try:
        response = requests.get(
            "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json",
            timeout=30
        )
        response.raise_for_status()
        kev_data = response.json()
        kev_cves = {vuln['cveID'] for vuln in kev_data.get('vulnerabilities', [])}
        print(f"   [OK] Loaded {len(kev_cves):,} KEV CVEs")
        
        # Update database
        kev_count = 0
        for cve_id in kev_cves:
            if cve_id in df['cve_id'].values:
                db.update_enrichment(cve_id, kev_flag=1)
                kev_count += 1
        print(f"   [OK] Flagged {kev_count:,} CVEs as KEV-listed")
    except Exception as e:
        print(f"   [WARN]  KEV fetch failed: {e}")
    
    # ========================================================================
    # 2. EPSS SCORES
    # ========================================================================
    print("\n[2/5] Fetching EPSS scores...")
    try:
        epss_fetcher = EPSSFetcher()
        cve_ids = df['cve_id'].tolist()
        
        # Batch processing
        batch_size = 100
        epss_count = 0
        for i in range(0, len(cve_ids), batch_size):
            batch = cve_ids[i:i+batch_size]
            scores = epss_fetcher.get_scores_bulk(batch)
            
            for cve_id, score_data in scores.items():
                db.update_enrichment(
                    cve_id,
                    epss_score=score_data.get('epss'),
                    epss_percentile=score_data.get('percentile'),
                    epss_date=score_data.get('date')
                )
                epss_count += 1
            
            if (i // batch_size + 1) % 10 == 0:
                print(f"   Progress: {i+len(batch):,}/{len(cve_ids):,} CVEs processed")
        
        print(f"   [OK] Added EPSS scores for {epss_count:,} CVEs")
    except Exception as e:
        print(f"   [WARN]  EPSS fetch failed: {e}")
    
    # ========================================================================
    # 3. HEALTHCARE MAPPING
    # ========================================================================
    print("\n[3/5] Healthcare keyword mapping...")
    try:
        mapper = HealthcareMapper()
        healthcare_count = 0
        
        for _, row in df.iterrows():
            is_hc, score = mapper.is_healthcare_related(
                description=row.get('description', ''),
                cwe=row.get('cwe', '')
            )
            if is_hc:
                db.update_enrichment(
                    row['cve_id'],
                    is_healthcare=1,
                    healthcare_score=score
                )
                healthcare_count += 1
        
        print(f"   [OK] Flagged {healthcare_count:,} healthcare-related CVEs")
    except Exception as e:
        print(f"   [WARN]  Healthcare mapping failed: {e}")
    
    # ========================================================================
    # 4. ATT&CK MAPPING
    # ========================================================================
    print("\n[4/5] MITRE ATT&CK mapping...")
    try:
        attack_mapper = AttackMapper()
        attack_count = 0
        
        for cve_id in df['cve_id']:
            techniques = attack_mapper.get_attack_techniques(cve_id)
            if techniques:
                db.update_enrichment(
                    cve_id,
                    attack_flag=1,
                    attack_technique_count=len(techniques)
                )
                attack_count += 1
        
        print(f"   [OK] Mapped {attack_count:,} CVEs to ATT&CK techniques")
    except Exception as e:
        print(f"   [WARN]  ATT&CK mapping failed: {e}")
    
    # ========================================================================
    # 5. CHPL MAPPING
    # ========================================================================
    print("\n[5/5] CHPL certified products mapping...")
    try:
        chpl_mapper = CHPLMapper()
        chpl_count = 0
        
        for _, row in df.iterrows():
            is_chpl = chpl_mapper.is_chpl_related(
                description=row.get('description', ''),
                cpe_list=row.get('cpe_list', [])
            )
            if is_chpl:
                db.update_enrichment(row['cve_id'], chpl_flag=1)
                chpl_count += 1
        
        print(f"   [OK] Flagged {chpl_count:,} CHPL-related CVEs")
    except Exception as e:
        print(f"   [WARN]  CHPL mapping failed: {e}")
    
    db.conn.close()
    
    print("\n" + "="*80)
    print("[OK] ENRICHMENT COMPLETE")
    print("="*80)

print("Enrichment pipeline loaded. Run: enrich_all_cves()")

Enrichment pipeline loaded. Run: enrich_all_cves()


In [7]:
# Uncomment to run full enrichment
# enrich_all_cves()

## 6. Validation & Quality Checks

In [11]:
def validate_data_quality():
    """Comprehensive data quality validation"""
    
    db_path = project_root / "data" / "cve_database.db"
    db = CVEDatabase(db_path)
    cursor = db.conn.cursor()
    
    print("="*80)
    print("DATA QUALITY VALIDATION")
    print("="*80)
    
    # CVE counts
    cursor.execute('SELECT COUNT(*) FROM cves')
    total_cves = cursor.fetchone()[0]
    
    cursor.execute('SELECT COUNT(*) FROM enrichments')
    enriched_cves = cursor.fetchone()[0]
    
    print(f"\n[STATS] Coverage:")
    print(f"   Total CVEs: {total_cves:,}")
    print(f"   Enriched: {enriched_cves:,} ({enriched_cves/total_cves*100:.1f}%)")
    
    # Enrichment signals
    print(f"\n[TARGET] Enrichment Signals:")
    
    signals = [
        ('KEV (exploited)', 'kev_flag'),
        ('Healthcare-related', 'is_healthcare'),
        ('ATT&CK mapped', 'attack_flag'),
        ('CHPL certified', 'chpl_flag'),
        ('Curated breaches', 'is_curated')
    ]
    
    for label, field in signals:
        cursor.execute(f'SELECT COUNT(*) FROM enrichments WHERE {field} = 1')
        count = cursor.fetchone()[0]
        pct = (count / enriched_cves * 100) if enriched_cves > 0 else 0
        print(f"   {label:30s} {count:6,} ({pct:5.2f}%)")
    
    # EPSS coverage
    cursor.execute('SELECT COUNT(*) FROM enrichments WHERE epss_score IS NOT NULL')
    epss_count = cursor.fetchone()[0]
    epss_pct = (epss_count / enriched_cves * 100) if enriched_cves > 0 else 0
    print(f"   {'EPSS scores':30s} {epss_count:6,} ({epss_pct:5.2f}%)")
    
    # Multi-signal CVEs
    print(f"\n High-Value CVEs (multiple signals):")
    
    cursor.execute('''
        SELECT COUNT(*) FROM enrichments 
        WHERE kev_flag = 1 AND is_healthcare = 1
    ''')
    kev_hc = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(*) FROM enrichments 
        WHERE chpl_flag = 1 AND is_healthcare = 1
    ''')
    chpl_hc = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(*) FROM enrichments 
        WHERE attack_flag = 1 AND is_healthcare = 1
    ''')
    attack_hc = cursor.fetchone()[0]
    
    cursor.execute('''
        SELECT COUNT(*) FROM enrichments 
        WHERE kev_flag = 1 AND attack_flag = 1 AND is_healthcare = 1
    ''')
    triple = cursor.fetchone()[0]
    
    print(f"   KEV + Healthcare: {kev_hc:,}")
    print(f"   CHPL + Healthcare: {chpl_hc:,}")
    print(f"   ATT&CK + Healthcare: {attack_hc:,}")
    print(f"   KEV + ATT&CK + Healthcare: {triple:,}")
    
    # Date distribution
    cursor.execute('''
        SELECT 
            strftime('%Y', published) as year,
            COUNT(*) as count
        FROM cves
        GROUP BY year
        ORDER BY year DESC
        LIMIT 10
    ''')
    years = cursor.fetchall()
    
    print(f"\n CVE Distribution by Year (last 10):")
    for year, count in years:
        print(f"   {year}: {count:,}")
    
    # CVSS distribution
    cursor.execute('''
        SELECT 
            CASE 
                WHEN cvss >= 9.0 THEN 'Critical (9.0-10.0)'
                WHEN cvss >= 7.0 THEN 'High (7.0-8.9)'
                WHEN cvss >= 4.0 THEN 'Medium (4.0-6.9)'
                ELSE 'Low (0.0-3.9)'
            END as severity,
            COUNT(*) as count
        FROM cves
        WHERE cvss IS NOT NULL
        GROUP BY severity
        ORDER BY MIN(cvss) DESC
    ''')
    severities = cursor.fetchall()
    
    print(f"\n[WARN]  CVSS Severity Distribution:")
    for severity, count in severities:
        pct = (count / total_cves * 100)
        print(f"   {severity:25s} {count:6,} ({pct:5.2f}%)")
    
    db.conn.close()
    
    print("\n" + "="*80)
    print("[OK] VALIDATION COMPLETE")
    print("="*80)

# Run validation
validate_data_quality()

2026-02-26 22:58:34 - src.core.cve_database - INFO - Connected to database
2026-02-26 22:58:34 - src.core.cve_database - INFO - Database schema created/verified
DATA QUALITY VALIDATION

[STATS] Coverage:
   Total CVEs: 226,320
   Enriched: 226,320 (100.0%)

[TARGET] Enrichment Signals:
   KEV (exploited)                 1,179 ( 0.52%)
   Healthcare-related                822 ( 0.36%)
   ATT&CK mapped                  83,574 (36.93%)
   CHPL certified                  5,107 ( 2.26%)
   Curated breaches                   52 ( 0.02%)
   EPSS scores                    226,320 (100.00%)

 High-Value CVEs (multiple signals):
   KEV + Healthcare: 2
   CHPL + Healthcare: 71
   ATT&CK + Healthcare: 426
   KEV + ATT&CK + Healthcare: 0

 CVE Distribution by Year (last 10):
   2025: 49,972
   2024: 40,704
   2023: 30,949
   2022: 26,431
   2021: 21,950
   2020: 19,222
   2019: 18,938
   2018: 18,154

[WARN]  CVSS Severity Distribution:
   Critical (9.0-10.0)       25,231 (11.15%)
   High (7.0-8.9)

## 7. Quick Analysis Queries

In [14]:
# Healthcare CVE Analysis Queries
db_path = project_root / "data" / "cve_database.db"
conn = sqlite3.connect(db_path)

# Query 1: KEV + Healthcare (Critical!)
print("=" * 100)
print("CRITICAL: KEV-Listed Healthcare CVEs (Actively Exploited)")
print("=" * 100)
query1 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.kev_flag = 1 AND e.is_healthcare = 1
ORDER BY c.cvss DESC, e.epss_score DESC
"""
kev_healthcare = pd.read_sql_query(query1, conn)
print(f"Found {len(kev_healthcare)} KEV + Healthcare CVEs\n")
display(kev_healthcare)

# Query 2: Top 20 Healthcare CVEs by CVSS
print("\n" + "=" * 100)
print("TOP 20: Highest Severity Healthcare CVEs")
print("=" * 100)
query2 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.chpl_flag,
    e.kev_flag,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.is_healthcare = 1
ORDER BY c.cvss DESC, e.healthcare_score DESC
LIMIT 20
"""
top_healthcare = pd.read_sql_query(query2, conn)
display(top_healthcare)

# Query 3: CHPL Certified Product CVEs
print("\n" + "=" * 100)
print("TOP 20: CHPL Certified Health IT Product Vulnerabilities")
print("=" * 100)
query3 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.kev_flag,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.chpl_flag = 1
ORDER BY c.cvss DESC, e.epss_score DESC
LIMIT 20
"""
chpl_cves = pd.read_sql_query(query3, conn)
print(f"Showing top 20 of {len(chpl_cves)} CHPL-related CVEs\n")
display(chpl_cves)

# Query 4: Multi-Signal Healthcare CVEs
print("\n" + "=" * 100)
print("HIGH VALUE: Healthcare CVEs with Multiple Risk Signals")
print("=" * 100)
query4 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.kev_flag,
    e.chpl_flag,
    e.attack_flag,
    CASE 
        WHEN e.kev_flag + e.chpl_flag + e.attack_flag >= 2 THEN 'High'
        ELSE 'Medium'
    END as priority,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.is_healthcare = 1 
  AND (e.kev_flag = 1 OR e.chpl_flag = 1 OR e.attack_flag = 1)
ORDER BY (e.kev_flag + e.chpl_flag + e.attack_flag) DESC, c.cvss DESC
LIMIT 25
"""
multi_signal = pd.read_sql_query(query4, conn)
display(multi_signal)

conn.close()

CRITICAL: KEV-Listed Healthcare CVEs (Actively Exploited)
Found 2 KEV + Healthcare CVEs



,cve_id,published,cvss,epss_score,healthcare_score,description_short
0,CVE-2023-43208,2023-10-26T17:15:09.033,9.8,0.0,0.7,NextGen Healthcare Mirth Connect before versio...
1,CVE-2020-10181,2020-03-11T16:15:12.007,9.8,0.0,0.5,goform/formEMR30 in Sumavision Enhanced Multim...



TOP 20: Highest Severity Healthcare CVEs


,cve_id,published,cvss,epss_score,healthcare_score,chpl_flag,kev_flag,description_short
0,CVE-2025-52572,2025-06-24T21:15:25.463,10.0,0.0,0.5,0,0,"Hikka, a Telegram userbot, has vulnerability a..."
1,CVE-2025-29009,2025-07-16T12:15:24.680,10.0,0.0,0.5,0,0,Unrestricted Upload of File with Dangerous Typ...
2,CVE-2025-22609,2025-01-24T17:15:15.100,10.0,0.0,0.5,0,0,Coolify is an open-source and self-hostable to...
3,CVE-2024-48967,2024-11-14T22:15:17.927,10.0,0.0,0.5,0,0,The ventilator and the Service PC lack suffici...
4,CVE-2024-48966,2024-11-14T22:15:17.727,10.0,0.0,0.5,0,0,The software tools used by service personnel t...
5,CVE-2019-5644,2019-11-06T19:15:12.547,10.0,0.0,0.5,0,0,Computing For Good's Basic Laboratory Informat...
6,CVE-2019-5617,2019-11-06T19:15:12.233,10.0,0.0,0.5,0,0,Computing For Good's Basic Laboratory Informat...
7,CVE-2019-10959,2019-06-13T21:29:15.817,10.0,0.0,0.5,0,0,"BD Alaris Gateway Workstation Versions, 1.1.3 ..."
8,CVE-2025-42967,2025-07-08T01:15:23.787,9.9,0.0,0.5,0,0,SAP S/4HANA and SAP SCM Characteristic Propaga...
9,CVE-2025-22611,2025-01-24T17:15:15.410,9.9,0.0,0.5,0,0,Coolify is an open-source and self-hostable to...



TOP 20: CHPL Certified Health IT Product Vulnerabilities
Showing top 20 of 20 CHPL-related CVEs



,cve_id,published,cvss,epss_score,healthcare_score,kev_flag,description_short
0,CVE-2025-2857,2025-03-27T14:15:55.720,10.0,0.0,0.0,0,Following the recent Chrome sandbox escape (CV...
1,CVE-2025-24786,2025-02-06T19:15:20.067,10.0,0.0,0.0,0,WhoDB is an open source database management to...
2,CVE-2025-20309,2025-07-02T17:15:52.927,10.0,0.0,0.0,0,A vulnerability in Cisco Unified Communication...
3,CVE-2025-10264,2025-09-12T10:15:31.640,10.0,0.0,0.0,0,Certain models of NVR developed by Digiever ha...
4,CVE-2024-49327,2024-10-20T09:15:04.440,10.0,0.0,0.0,0,Unrestricted Upload of File with Dangerous Typ...
5,CVE-2024-42472,2024-08-15T19:15:19.233,10.0,0.0,0.0,0,Flatpak is a Linux application sandboxing and ...
6,CVE-2024-37228,2024-06-24T13:15:10.947,10.0,0.0,0.0,0,Improper Control of Generation of Code ('Code ...
7,CVE-2024-2973,2024-06-27T21:15:15.037,10.0,0.0,0.0,0,An Authentication Bypass Using an Alternate Pa...
8,CVE-2023-6248,2023-11-21T22:15:08.787,10.0,0.0,0.0,0,The Syrus4 IoT gateway utilizes an unsecured M...
9,CVE-2023-42454,2023-09-18T22:15:47.547,10.0,0.0,0.0,0,SQLpage is a SQL-only webapp builder. Someone ...



HIGH VALUE: Healthcare CVEs with Multiple Risk Signals


,cve_id,published,cvss,epss_score,healthcare_score,kev_flag,chpl_flag,attack_flag,priority,description_short
0,CVE-2024-36543,2024-06-17T19:15:58.353,9.8,0.0,0.5,0,1,1,High,Incorrect access control in the Kafka Connect ...
1,CVE-2023-43208,2023-10-26T17:15:09.033,9.8,0.0,0.7,1,1,0,High,NextGen Healthcare Mirth Connect before versio...
2,CVE-2024-48970,2024-11-14T22:15:18.137,9.3,0.0,0.5,0,1,1,High,The ventilator's microcontroller lacks memory ...
3,CVE-2019-11875,2019-05-24T16:29:00.437,8.8,0.0,0.5,0,1,1,High,In AutomateAppCore.dll in Blue Prism Robotic P...
4,CVE-2017-9388,2019-06-17T17:15:10.537,8.8,0.0,0.5,0,1,1,High,An issue was discovered on Vera VeraEdge 1.7.1...
5,CVE-2017-9384,2019-06-17T18:15:10.627,8.8,0.0,0.5,0,1,1,High,An issue was discovered on Vera VeraEdge 1.7.1...
6,CVE-2017-12712,2018-04-25T13:29:00.227,8.8,0.0,0.8,0,1,1,High,The authentication algorithm in Abbott Laborat...
7,CVE-2025-43860,2025-05-23T16:15:25.620,7.6,0.0,0.5,0,1,1,High,OpenEMR is a free and open source electronic h...
8,CVE-2025-32794,2025-05-23T16:15:25.300,7.6,0.0,0.5,0,1,1,High,OpenEMR is a free and open source electronic h...
9,CVE-2025-31117,2025-03-31T17:15:42.833,7.5,0.0,0.5,0,1,1,High,OpenEMR is a free and open source electronic h...


## 8. Export Data for Analysis

In [13]:
def export_enriched_data(output_path: str = None):
    """Export enriched CVE data to CSV"""
    
    if output_path is None:
        output_path = project_root / "outputs" / f"enriched_cves_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    db_path = project_root / "data" / "cve_database.db"
    conn = sqlite3.connect(db_path)
    
    query = """
    SELECT 
        c.*,
        e.kev_flag,
        e.epss_score,
        e.epss_percentile,
        e.is_healthcare,
        e.healthcare_score,
        e.attack_flag,
        e.attack_technique_count,
        e.chpl_flag,
        e.is_curated,
        e.label
    FROM cves c
    LEFT JOIN enrichments e ON c.cve_id = e.cve_id
    """
    
    df = pd.read_sql_query(query, conn)
    conn.close()
    
    df.to_csv(output_path, index=False)
    print(f"[OK] Exported {len(df):,} CVEs to: {output_path}")
    
    return df

# Uncomment to export
# export_enriched_data()

## 9. Summary

**Complete Workflow**:

1. **Check status**: `check_current_status()`
2. **Reset (if needed)**: `reset_all(confirm=True)`
3. **Fetch CVEs**: `fetch_cves_by_date('2024-01-01', '2024-12-31')`
4. **Enrich data**: `enrich_all_cves()`
5. **Validate**: `validate_data_quality()`
6. **Export**: `export_enriched_data()`

**Next Steps**:
- Feature engineering notebook
- Model training notebook
- Recommendation generation

---